# Notebook 00 â€” Colab Setup (Helper, bukan bagian dari 10 notebook utama)

Jalankan notebook ini **pertama kali** di Google Colab sebelum notebook 01â€“10.
Di lokal (Windows), lewati notebook ini â€” data sudah ada di `D:/data viewer/data/raw/`.

Isi:
1. Install dependencies
2. Buat struktur folder `/content/solar-review/`
3. Download dataset publik (NAB, DeepMatcher, HoloClean, UCI Donation, FEBRL4)
4. Panduan download Solar Power Generation (butuh `kaggle.json`)
5. Verifikasi semua file

In [ ]:
# 1. Install dependencies
!pip -q install recordlinkage scikit-learn pandas matplotlib seaborn openpyxl kaggle

import os
BASE = "/content/solar-review"
RAW = f"{BASE}/data/raw"
OUT = f"{BASE}/data/processed"
REP = f"{BASE}/reports"
for d in [RAW, OUT, REP]:
    os.makedirs(d, exist_ok=True)
print("BASE:", BASE)

In [ ]:
# 2. Download dataset publik tanpa login (NAB + DeepMatcher + HoloClean)
import requests

jobs = [
    ("numenta/NAB", "data/realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv", "nab/realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv"),
    ("numenta/NAB", "data/realAWSCloudwatch/grok_asg_anomaly.csv", "nab/realAWSCloudwatch/grok_asg_anomaly.csv"),
    ("numenta/NAB", "data/realKnownCause/machine_temperature_system_failure.csv", "nab/realKnownCause/machine_temperature_system_failure.csv"),
    ("numenta/NAB", "data/realKnownCause/nyc_taxi.csv", "nab/realKnownCause/nyc_taxi.csv"),
    ("numenta/NAB", "labels/combined_windows.json", "nab/labels/combined_windows.json"),
    ("icip-cas/EntityMatcher", "data/dirty_walmart_amazon/train.csv", "deepmatcher/dirty_walmart_amazon/train.csv"),
    ("icip-cas/EntityMatcher", "data/dirty_walmart_amazon/test.csv", "deepmatcher/dirty_walmart_amazon/test.csv"),
    ("icip-cas/EntityMatcher", "data/walmart_amazon/test.csv", "deepmatcher/walmart_amazon/test.csv"),
    ("icip-cas/EntityMatcher", "data/dirty_dblp_acm/test.csv", "deepmatcher/dirty_dblp_acm/test.csv"),
    ("HoloClean/holoclean", "testdata/hospital.csv", "holoclean/hospital.csv"),
    ("HoloClean/holoclean", "testdata/hospital_clean.csv", "holoclean/hospital_clean.csv"),
    ("HoloClean/holoclean", "testdata/hospital_constraints.txt", "holoclean/hospital_constraints.txt"),
    ("HoloClean/holoclean", "testdata/flight.csv", "holoclean/flight.csv"),
    ("HoloClean/holoclean", "testdata/flight_clean.csv", "holoclean/flight_clean.csv"),
]

for repo, src, dst in jobs:
    out = f"{RAW}/{dst}"
    if os.path.exists(out):
        print("SKIP (exists):", dst)
        continue
    os.makedirs(os.path.dirname(out), exist_ok=True)
    r = requests.get(f"https://raw.githubusercontent.com/{repo}/master/{src}", timeout=120)
    assert r.status_code == 200 and len(r.content) > 500, f"download gagal: {dst}"
    open(out, "wb").write(r.content)
    print("OK:", dst, len(r.content))

In [ ]:
# 3. UCI Donation (record linkage, ~56 MB)
import urllib.request, zipfile

zip_path = f"{RAW}/donation.zip"
if not os.path.exists(f"{RAW}/donation/block_1.zip"):
    print("Downloading UCI donation.zip ...")
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00210/donation.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(f"{RAW}/donation")
    print("extracted")
else:
    print("SKIP (exists): donation")

# FEBRL4 (via package, tanpa download manual)
from recordlinkage.datasets import load_febrl4
df_a, df_b = load_febrl4()
print(f"FEBRL4: df_a={df_a.shape}, df_b={df_b.shape}")

In [ ]:
# 4. Solar Power Generation (butuh kaggle.json — upload sekali di Colab)
# Langkah: buka https://www.kaggle.com/settings -> API -> Create New Token -> upload file kaggle.json di bawah
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Bukan di Colab - lewati bagian ini (data solar sudah ada lokal).")
import shutil

if IN_COLAB:
    os.makedirs("/root/.kaggle", exist_ok=True)
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload file kaggle.json:")
        up = files.upload()  # pilih kaggle.json
        for name in up:
            shutil.move(name, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    get_ipython().system("mkdir -p /content/solar-review/data/raw/solar_power_generation && kaggle datasets download anikannal/solar-power-generation-data -p /content/solar-review/data/raw/solar_power_generation --unzip")
    print("Solar Power Generation: OK")


In [ ]:
# 5. Verifikasi + PATH SHIM
# PENTING: semua notebook memakai path Windows 'D:/data viewer'.
# Di Colab, jalankan find-replace:  D:/data viewer  ->  /content/solar-review
# (Edit -> Find and replace in notebook, atau ganti manual di cell pertama tiap notebook)
for root, _, fs in os.walk(RAW):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p):>10}  {os.path.relpath(p, RAW)}")